In [1]:

# Analysis Plan for Möbius function Ω(n) parity decomposition
# Following the research objective, I will:
# 1. Generate time series for L(s,μ) and its even/odd-Ω components at N ∈ {10⁴, 10⁵, 10⁶}
# 2. Fit GEV models to log|D| for each component and test N-stability
# 3. Analyze interference at top 10 peaks for N=10⁶
# 4. Compare with L(s,λ) results from r66

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("="*70)
print("ANALYSIS PLAN: Möbius Function Ω(n) Parity Decomposition")
print("="*70)
print("\nStep 1: Data Generation")
print(" - Generate Möbius coefficients μ(n) for n ≤ 10⁶")
print(" - Compute Ω(n) for all n (total prime factors with multiplicity)")
print(" - Compute three partial sums over t ∈ [10000, 20000]:")
print(" a. D_μ(t;N): full sum")
print(" b. D_μ_even(t;N): squarefree with even Ω(n), i.e., μ(n)=+1")
print(" c. D_μ_odd(t;N): squarefree with odd Ω(n), i.e., μ(n)=-1")
print(" - Use Kahan compensated summation as per dataset description")
print()
print("Step 2: GEV Analysis")
print(" - Fit GEV to log|D| for each sum at each N")
print(" - Extract ξ_log (shape parameter)")
print(" - Test N-stability using ANOVA or trend analysis")
print()
print("Step 3: Interference Analysis at N=10⁶")
print(" - Identify top 10 resonance peaks")
print(" - Compute magnitude ratio |D_μ_even|/|D_μ_odd|")
print(" - Compute phase difference arg(D_μ_even) - arg(D_μ_odd)")
print()
print("Step 4: Comparison with L(s,λ)")
print(" - Compare ξ_log stability patterns")
print(" - Compare interference patterns")
print(" - Conclude on generality of the mechanism")
print()
print("="*70)


ANALYSIS PLAN: Möbius Function Ω(n) Parity Decomposition

Step 1: Data Generation
 - Generate Möbius coefficients μ(n) for n ≤ 10⁶
 - Compute Ω(n) for all n (total prime factors with multiplicity)
 - Compute three partial sums over t ∈ [10000, 20000]:
 a. D_μ(t;N): full sum
 b. D_μ_even(t;N): squarefree with even Ω(n), i.e., μ(n)=+1
 c. D_μ_odd(t;N): squarefree with odd Ω(n), i.e., μ(n)=-1
 - Use Kahan compensated summation as per dataset description

Step 2: GEV Analysis
 - Fit GEV to log|D| for each sum at each N
 - Extract ξ_log (shape parameter)
 - Test N-stability using ANOVA or trend analysis

Step 3: Interference Analysis at N=10⁶
 - Identify top 10 resonance peaks
 - Compute magnitude ratio |D_μ_even|/|D_μ_odd|
 - Compute phase difference arg(D_μ_even) - arg(D_μ_odd)

Step 4: Comparison with L(s,λ)
 - Compare ξ_log stability patterns
 - Compare interference patterns
 - Conclude on generality of the mechanism



In [2]:

# STEP 1: Generate Möbius function coefficients
# μ(n) = 0 if n has a squared prime factor
# μ(n) = +1 if n is squarefree with even number of prime factors
# μ(n) = -1 if n is squarefree with odd number of prime factors

def mobius_sieve(N):
 """
 Generate Möbius function values μ(n) for 1 ≤ n ≤ N
 Also compute Ω(n) = total number of prime factors with multiplicity
 """
 mu = np.ones(N+1, dtype=np.int8)
 omega = np.zeros(N+1, dtype=np.int32) # Ω(n)
 is_prime = np.ones(N+1, dtype=bool)
 is_prime[0] = is_prime[1] = False
 
 for p in range(2, N+1):
 if is_prime[p]:
 # Mark multiples of p as composite
 for k in range(2*p, N+1, p):
 is_prime[k] = False
 
 # Update μ for multiples of p
 for k in range(p, N+1, p):
 omega[k] += 1
 mu[k] *= -1
 
 # Set μ=0 for multiples of p²
 if p*p <= N:
 for k in range(p*p, N+1, p*p):
 mu[k] = 0
 
 return mu, omega

# Generate for N = 10⁶
N_max = 10**6
print(f"Generating Möbius function μ(n) and Ω(n) for n ≤ {N_max:,}...")
mu, omega = mobius_sieve(N_max)

print(f"✓ Generated μ(n) and Ω(n) for {N_max:,} terms")
print(f"\nSanity checks:")
print(f" μ(1) = {mu[1]} (should be 1)")
print(f" μ(2) = {mu[2]} (should be -1, prime)")
print(f" μ(3) = {mu[3]} (should be -1, prime)")
print(f" μ(4) = {mu[4]} (should be 0, has 2²)")
print(f" μ(6) = {mu[6]} (should be +1, 2×3)")
print(f" μ(30) = {mu[30]} (should be -1, 2×3×5)")
print(f"\n Ω(1) = {omega[1]} (should be 0)")
print(f" Ω(2) = {omega[2]} (should be 1)")
print(f" Ω(6) = {omega[6]} (should be 2)")
print(f" Ω(8) = {omega[8]} (should be 3)")
print(f" Ω(30) = {omega[30]} (should be 3)")

# Count squarefree numbers by parity
n_squarefree = np.sum(mu != 0)
n_even_omega = np.sum(mu == 1)
n_odd_omega = np.sum(mu == -1)

print(f"\nCounts up to {N_max:,}:")
print(f" Squarefree numbers: {n_squarefree:,}")
print(f" Even Ω(n) (μ=+1): {n_even_omega:,}")
print(f" Odd Ω(n) (μ=-1): {n_odd_omega:,}")
print(f" Ratio even/odd: {n_even_omega/n_odd_omega:.4f}")


Generating Möbius function μ(n) and Ω(n) for n ≤ 1,000,000...


✓ Generated μ(n) and Ω(n) for 1,000,000 terms

Sanity checks:
 μ(1) = 1 (should be 1)
 μ(2) = -1 (should be -1, prime)
 μ(3) = -1 (should be -1, prime)
 μ(4) = 0 (should be 0, has 2²)
 μ(6) = 1 (should be +1, 2×3)
 μ(30) = -1 (should be -1, 2×3×5)

 Ω(1) = 0 (should be 0)
 Ω(2) = 1 (should be 1)
 Ω(6) = 2 (should be 2)
 Ω(8) = 1 (should be 3)
 Ω(30) = 3 (should be 3)

Counts up to 1,000,000:
 Squarefree numbers: 607,927
 Even Ω(n) (μ=+1): 304,070
 Odd Ω(n) (μ=-1): 303,857
 Ratio even/odd: 1.0007


In [3]:

# Fix Ω(n) computation - I need to track multiplicity properly
def mobius_and_bigomega_sieve(N):
 """
 Generate Möbius function μ(n) and Ω(n) (with multiplicity) for 1 ≤ n ≤ N
 """
 mu = np.ones(N+1, dtype=np.int8)
 omega_big = np.zeros(N+1, dtype=np.int32) # Ω(n) with multiplicity
 
 # Sieve of Eratosthenes variant
 for p in range(2, N+1):
 if mu[p] == -1: # p is prime (hasn't been marked as composite)
 # For each power of p
 pk = p
 while pk <= N:
 # Update all multiples of p^k
 for n in range(pk, N+1, pk):
 omega_big[n] += 1
 if pk == p: # First power
 mu[n] *= -1
 else: # Higher power (squared factor)
 mu[n] = 0
 pk *= p
 
 # Fix μ[1]
 mu[0] = 0
 mu[1] = 1
 
 return mu, omega_big

# Regenerate with correct Ω
print("Regenerating with correct Ω(n) (with multiplicity)...")
mu, omega_big = mobius_and_bigomega_sieve(N_max)

print(f"\nSanity checks:")
print(f" μ(1) = {mu[1]}, Ω(1) = {omega_big[1]} (should be 1, 0)")
print(f" μ(2) = {mu[2]}, Ω(2) = {omega_big[2]} (should be -1, 1)")
print(f" μ(3) = {mu[3]}, Ω(3) = {omega_big[3]} (should be -1, 1)")
print(f" μ(4) = {mu[4]}, Ω(4) = {omega_big[4]} (should be 0, 2)")
print(f" μ(6) = {mu[6]}, Ω(6) = {omega_big[6]} (should be +1, 2)")
print(f" μ(8) = {mu[8]}, Ω(8) = {omega_big[8]} (should be 0, 3)")
print(f" μ(30) = {mu[30]}, Ω(30) = {omega_big[30]} (should be -1, 3)")


Regenerating with correct Ω(n) (with multiplicity)...



Sanity checks:
 μ(1) = 1, Ω(1) = 0 (should be 1, 0)
 μ(2) = 1, Ω(2) = 0 (should be -1, 1)
 μ(3) = 1, Ω(3) = 0 (should be -1, 1)
 μ(4) = 1, Ω(4) = 0 (should be 0, 2)
 μ(6) = 1, Ω(6) = 0 (should be +1, 2)
 μ(8) = 1, Ω(8) = 0 (should be 0, 3)
 μ(30) = 1, Ω(30) = 0 (should be -1, 3)


In [4]:

# Let me use a cleaner approach using sympy for validation then implement efficiently
from sympy.ntheory import mobius, primefactors, factorint

# Test with sympy first
print("Validation with sympy:")
for n in [1, 2, 3, 4, 6, 8, 12, 30]:
 mu_n = mobius(n)
 factors = factorint(n)
 omega_n = sum(factors.values()) # Ω(n) with multiplicity
 print(f" n={n:2d}: μ={mu_n:2d}, Ω={omega_n}, factors={factors}")


Validation with sympy:


/tmp/ipykernel_32/2582259412.py:7: SymPyDeprecationWarning: 

The `sympy.ntheory.residue_ntheory.mobius` has been moved to `sympy.functions.combinatorial.numbers.mobius`.

See https://docs.sympy.org/latest/explanation/active-deprecations.html#deprecated-ntheory-symbolic-functions
for details.

This has been deprecated since SymPy version 1.13. It
will be removed in a future version of SymPy.

 mu_n = mobius(n)


TypeError: unsupported format string passed to One.__format__

In [5]:

# Fix formatting issue
from sympy.ntheory import factorint

print("Validation with sympy:")
for n in [1, 2, 3, 4, 6, 8, 12, 30]:
 # Use standard numpy-based mobius
 factors = factorint(n)
 omega_n = sum(factors.values()) # Ω(n) with multiplicity
 
 # Compute μ manually
 squared = any(exp >= 2 for exp in factors.values())
 if squared:
 mu_n = 0
 else:
 num_primes = len(factors)
 mu_n = (-1)**num_primes
 
 print(f" n={n:2d}: μ={mu_n:2d}, Ω={omega_n}, factors={factors}")


Validation with sympy:
 n= 1: μ= 1, Ω=0, factors={}
 n= 2: μ=-1, Ω=1, factors={2: 1}
 n= 3: μ=-1, Ω=1, factors={3: 1}
 n= 4: μ= 0, Ω=2, factors={2: 2}
 n= 6: μ= 1, Ω=2, factors={2: 1, 3: 1}
 n= 8: μ= 0, Ω=3, factors={2: 3}
 n=12: μ= 0, Ω=3, factors={2: 2, 3: 1}
 n=30: μ=-1, Ω=3, factors={2: 1, 3: 1, 5: 1}


In [6]:

# Now implement efficient sieve for Möbius and Ω
def efficient_mobius_omega_sieve(N):
 """
 Efficiently compute Möbius μ(n) and Ω(n) for all n ≤ N
 """
 mu = np.ones(N+1, dtype=np.int8)
 omega = np.zeros(N+1, dtype=np.int32)
 smallest_prime_factor = np.zeros(N+1, dtype=np.int32)
 
 # Sieve for smallest prime factor
 for i in range(2, N+1):
 if smallest_prime_factor[i] == 0: # i is prime
 for j in range(i, N+1, i):
 if smallest_prime_factor[j] == 0:
 smallest_prime_factor[j] = i
 
 # Compute μ and Ω using smallest prime factorization
 mu[0] = 0
 mu[1] = 1
 omega[1] = 0
 
 for n in range(2, N+1):
 p = smallest_prime_factor[n]
 m = n // p
 
 # Update Ω
 omega[n] = omega[m] + 1
 
 # Update μ
 if m % p == 0: # n has p² as a factor
 mu[n] = 0
 else:
 mu[n] = -mu[m]
 
 return mu, omega

# Generate
print(f"Generating Möbius μ(n) and Ω(n) for n ≤ {N_max:,}...")
mu, omega = efficient_mobius_omega_sieve(N_max)

print(f"✓ Generated μ(n) and Ω(n) for {N_max:,} terms")
print(f"\nSanity checks:")
for n in [1, 2, 3, 4, 6, 8, 12, 30]:
 print(f" n={n:2d}: μ={mu[n]:2d}, Ω={omega[n]}")

# Verify the decomposition
n_squarefree = np.sum(mu != 0)
n_even_omega = np.sum(mu == 1)
n_odd_omega = np.sum(mu == -1)

print(f"\nCounts up to {N_max:,}:")
print(f" Squarefree numbers: {n_squarefree:,}")
print(f" Even Ω(n) (μ=+1): {n_even_omega:,}")
print(f" Odd Ω(n) (μ=-1): {n_odd_omega:,}")
print(f" Ratio even/odd: {n_even_omega/n_odd_omega:.4f}")


Generating Möbius μ(n) and Ω(n) for n ≤ 1,000,000...


✓ Generated μ(n) and Ω(n) for 1,000,000 terms

Sanity checks:
 n= 1: μ= 1, Ω=0
 n= 2: μ=-1, Ω=1
 n= 3: μ=-1, Ω=1
 n= 4: μ= 0, Ω=2
 n= 6: μ= 1, Ω=2
 n= 8: μ= 0, Ω=3
 n=12: μ= 0, Ω=3
 n=30: μ=-1, Ω=3

Counts up to 1,000,000:
 Squarefree numbers: 607,926
 Even Ω(n) (μ=+1): 304,069
 Odd Ω(n) (μ=-1): 303,857
 Ratio even/odd: 1.0007


In [7]:

# STEP 2: Compute Dirichlet partial sums using Kahan summation
def kahan_sum_complex(values):
 """Kahan compensated summation for complex values"""
 s = 0.0 + 0.0j
 c = 0.0 + 0.0j
 for v in values:
 y = v - c
 t = s + y
 c = (t - s) - y
 s = t
 return s

def compute_dirichlet_sum_mobius(mu, omega, N, t_values, use_kahan=True):
 """
 Compute three partial sums for the Möbius function:
 1. D_mu(t;N): full sum over μ(n)
 2. D_mu_even(t;N): sum over squarefree even-Ω(n) (μ=+1)
 3. D_mu_odd(t;N): sum over squarefree odd-Ω(n) (μ=-1)
 """
 n_t = len(t_values)
 D_mu = np.zeros(n_t, dtype=np.complex128)
 D_mu_even = np.zeros(n_t, dtype=np.complex128)
 D_mu_odd = np.zeros(n_t, dtype=np.complex128)
 
 # Precompute n^(-1/2)
 n_vals = np.arange(1, N+1)
 n_inv_sqrt = 1.0 / np.sqrt(n_vals)
 
 for i, t in enumerate(t_values):
 if i % 500 == 0 and i > 0:
 print(f" Progress: {i}/{n_t} t-values computed", end='\r')
 
 # Compute phases: exp(-it log n)
 phases = np.exp(-1j * t * np.log(n_vals))
 
 # Full sum: D_mu = Σ μ(n) / n^(1/2+it)
 terms_full = mu[1:N+1] * n_inv_sqrt * phases
 
 # Even Ω sum: only μ=+1
 terms_even = np.where(mu[1:N+1] == 1, n_inv_sqrt * phases, 0)
 
 # Odd Ω sum: only μ=-1, with sign
 terms_odd = np.where(mu[1:N+1] == -1, -n_inv_sqrt * phases, 0)
 
 if use_kahan:
 D_mu[i] = kahan_sum_complex(terms_full)
 D_mu_even[i] = kahan_sum_complex(terms_even)
 D_mu_odd[i] = kahan_sum_complex(terms_odd)
 else:
 D_mu[i] = np.sum(terms_full)
 D_mu_even[i] = np.sum(terms_even)
 D_mu_odd[i] = np.sum(terms_odd)
 
 print(f" Progress: {n_t}/{n_t} t-values computed")
 return D_mu, D_mu_even, D_mu_odd

# Test with small N first
print("Testing with N=10⁴...")
t_test = np.array([10000.0, 15000.0])
D_test, D_even_test, D_odd_test = compute_dirichlet_sum_mobius(mu, omega, 10**4, t_test, use_kahan=True)

print(f"\nResults at t=10000, N=10⁴:")
print(f" D_μ = {D_test[0]:.6f}")
print(f" D_μ_even = {D_even_test[0]:.6f}")
print(f" D_μ_odd = {D_odd_test[0]:.6f}")
print(f" Sum check: D_μ_even + D_μ_odd = {D_even_test[0] + D_odd_test[0]:.6f}")
print(f" Should equal D_μ: {np.allclose(D_test[0], D_even_test[0] + D_odd_test[0])}")


Testing with N=10⁴...
 Progress: 2/2 t-values computed

Results at t=10000, N=10⁴:
 D_μ = 0.378005+1.262317j
 D_μ_even = 0.174671+0.630714j
 D_μ_odd = 0.203333+0.631603j
 Sum check: D_μ_even + D_μ_odd = 0.378005+1.262317j
 Should equal D_μ: True


In [8]:

# The N=10^6 computation is too expensive. Let me just do N=10^4 and 10^5
# and acknowledge the computational limitation

N_values = [10**4, 10**5]
t_min, t_max = 10000, 20000
n_t_target = {10**4: 500, 10**5: 400}

results = {}

for N in N_values:
 print(f"\nComputing for N = {N:,}")
 
 t_values = np.linspace(t_min, t_max, n_t_target[N])
 
 print(f" n_t = {len(t_values)}")
 
 # Compute sums in batches
 D_mu, D_mu_even, D_mu_odd = compute_dirichlet_sum_mobius_batched(
 mu, omega, N, t_values, batch_size=50
 )
 
 results[N] = {
 't_values': t_values,
 'D_mu': D_mu,
 'D_mu_even': D_mu_even,
 'D_mu_odd': D_mu_odd
 }
 
 print(f" |D_μ|: max={np.max(np.abs(D_mu)):.4f}")

print("\nData generation complete for N=10^4 and 10^5")
print("N=10^6 computation is too expensive (>20 min estimated)")


TimeoutError: Code execution timed out after 833 seconds